In [11]:
# TODO: make functions less verbose for data creation
# TODO: clean import statments
# TODO: see if we are calculating Total Energy correctly, I think the whole row is all 0 after std so ... 
# TODO: Experiment with the derivatives and  not derivatives data (preprocessed and 'raw' respectively, I think currently raw_

In [12]:
 # this is a good little tutorial to understand basics of pyspark  
# https://domino.ai/blog/principal-component-analysis-pca-on-large-neuroimaging-datasets-using-pyspark

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [14]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

In [15]:
# Spark is a library that distributes the load of computation/ram very efficiently and evenly :)

In [16]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

Stopping existing Spark context...
Previous Spark context stopped successfully


In [17]:
# Set environment variables
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# Create new session with explicit local binding
spark = SparkSession.builder \
    .appName("EEG_Analysis") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .master("local[*]") \
    .getOrCreate()


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

New Spark session created successfully


25/04/07 23:33:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [18]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context


In [19]:
subject_df = load_subjects_df(spark, participants_path="/Users/user/eeg-ds004504/ds004504/participants.tsv") #this is the .tsv with the information of all the participants

In [20]:
%%time
# we need to give the path of our  data directory to process the EEG data from
from preprocess_sets import set_data_path, get_data_path

# set_data_path("/Users/user/eeg-ds004504") !!! this doesn't work! so we need to do it manually in preprocess_sets.py! or else won't work!
print(get_data_path())

#Example below is how to get a single subject  and extract its features
sub1 = (
    subject_df
    .filter((subject_df.SubjectID == "sub-001"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)
sub1.show()

/Users/user/eeg-ds004504


/Users/user/jupyter-venv/lib/python3.12/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
Processing subject sub-001                                          (0 + 1) / 1]
processSub sub-001
subPath sub-001
subPath DATA_PATH /Users/user/eeg-ds004504/
Path handed: /Users/user/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Got 398 epochs for sub-001
<class 'mne.epochs.Epochs'>
Epoch 0
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>
Epoch 1
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>
Total rows collected: 37810
Returning DataFrame with 37810 rows
Column names from schema: ['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']
18.43074893951416
                                                                                

+---------+-------+--------+---------+--------------------+
|SubjectID|EpochID|WaveBand|Electrode|               Power|
+---------+-------+--------+---------+--------------------+
|  sub-001|   ep-0|   Delta|      Fp1| 0.08257892642542036|
|  sub-001|   ep-0|   Theta|      Fp1|0.005611645685714301|
|  sub-001|   ep-0|   Alpha|      Fp1|0.001143220388171...|
|  sub-001|   ep-0|    Beta|      Fp1|1.958040080323614...|
|  sub-001|   ep-0|   Total|      Fp1|0.011235955056179778|
|  sub-001|   ep-0|   Delta|      Fp2| 0.08389220643958978|
|  sub-001|   ep-0|   Theta|      Fp2|0.004672355992737973|
|  sub-001|   ep-0|   Alpha|      Fp2|9.638188967593915E-4|
|  sub-001|   ep-0|    Beta|      Fp2|1.768820461211869...|
|  sub-001|   ep-0|   Total|      Fp2|0.011235955056179771|
|  sub-001|   ep-0|   Delta|       F3| 0.08212274846138011|
|  sub-001|   ep-0|   Theta|       F3|0.005595318986261473|
|  sub-001|   ep-0|   Alpha|       F3|0.001112336388491...|
|  sub-001|   ep-0|    Beta|       F3|2.

In [21]:
%%time

# this is the magic of pyspark's distributed system: What would take 50 minutes single threaded takes around 4.5
# I did a similiar optimization with joblib where I would multiprocess this steap and it would take around 7

# What's nice is that we can process whole groups pretty easily :)
group_a_spark_df = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

group_c_spark_df = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)


# Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
result_group_a = group_a_spark_df.persist()
result_group_c = group_c_spark_df.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")

Processing subject sub-016============================>        (168 + 12) / 200]
processSub sub-016
Processing subject sub-002
processSub sub-002
Processing subject sub-012
processSub sub-012
Processing subject sub-001
processSub sub-001
Processing subject sub-011
processSub sub-011
Processing subject sub-022
processSub sub-022
Processing subject sub-020
processSub sub-020
Processing subject sub-021
processSub sub-021
Processing subject sub-017
processSub sub-017
Processing subject sub-009
processSub sub-009
Processing subject sub-018
processSub sub-018
Processing subject sub-029
processSub sub-029
subPath sub-016
subPath DATA_PATH /Users/user/eeg-ds004504/
Path handed: /Users/user/eeg-ds004504/ds004504/sub-016/eeg/sub-016_task-eyesclosed_eeg.set
subPath sub-002
subPath DATA_PATH /Users/user/eeg-ds004504/
Path handed: /Users/user/eeg-ds004504/ds004504/sub-002/eeg/sub-002_task-eyesclosed_eeg.set
subPath sub-012
subPath DATA_PATH /Users/user/eeg-ds004504/
Path handed: /Users/user/eeg-ds0

Processed 1856870 records for Alzheimer's group
Processed 1543465 records for Control group
CPU times: user 1.23 s, sys: 453 ms, total: 1.68 s
Wall time: 16min 54s


In [22]:
result_group_a.columns

['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']

In [23]:
#Since we doni't want to recreate the data all the time, lets save it and I will see you in Example_Data_Processing

In [24]:
type(group_a_spark_df)

pyspark.sql.dataframe.DataFrame

In [25]:
group_a_pandas_df = group_a_spark_df.toPandas() # see here, spark has its own data frame type with lots of its own functions
group_c_pandas_df = group_c_spark_df.toPandas() # Each .pkl is around 50mb last time I checked


In [26]:
group_a_pandas_df.to_pickle("features_alz_example.pkl") # pkl is a way to store python dataframes, its nice
group_c_pandas_df.to_pickle("features_cntrl_example.pkl") # pkl is a way to store python dataframes, its nice

In [27]:
# This is how we would load the .pkl's back in 
# Step 1: Load back into pandas
group_a_pandas_df_loaded = pd.read_pickle("features_alz_example.pkl")
group_c_pandas_df_loaded = pd.read_pickle("features_cntrl_example.pkl")

# Step 2: Convert to Spark DataFrames
group_a_spark_df_loaded = spark.createDataFrame(group_a_pandas_df_loaded)
group_c_spark_df_loaded = spark.createDataFrame(group_c_pandas_df_loaded)

In [28]:
type(group_a_spark_df)

pyspark.sql.dataframe.DataFrame

In [29]:
if group_a_spark_df_loaded.exceptAll(group_a_spark_df).isEmpty(): 
    print("Correctly pkl'd and correcfly loaded into pyspark object")

25/04/07 23:53:44 WARN TaskSetManager: Stage 24 contains a task of very large size (6612 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Correctly pkl'd and correcfly loaded into pyspark object


In [30]:
group_c_spark_df_loaded.select("SubjectID").distinct().orderBy("SubjectID").show(truncate=False)

25/04/07 23:53:47 WARN TaskSetManager: Stage 34 contains a task of very large size (5470 KiB). The maximum recommended task size is 1000 KiB.
[Stage 34:>                                                       (0 + 12) / 12]

+---------+
|SubjectID|
+---------+
|sub-037  |
|sub-038  |
|sub-039  |
|sub-040  |
|sub-041  |
|sub-042  |
|sub-043  |
|sub-044  |
|sub-045  |
|sub-046  |
|sub-047  |
|sub-048  |
|sub-049  |
|sub-050  |
|sub-051  |
|sub-052  |
|sub-053  |
|sub-054  |
|sub-055  |
|sub-056  |
+---------+
only showing top 20 rows

